<a href="https://www.kaggle.com/code/drsairaakram/ml-models?scriptVersionId=364409665" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

# 2. Train / Val / Test Split (70% Train, 15% Validation, 15% Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# 3. Preprocessing: Scaling (FIT ONLY ON TRAINING DATA!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)      # Only transform!
X_test_scaled = scaler.transform(X_test)    # Only transform!

# 4. Dummy Baseline (Predicts the mean of y_train)
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)
dummy_rmse = np.sqrt(mean_squared_error(y_test, y_pred_dummy))
print(f"\n--- Baseline (Dummy Regressor) RMSE: {dummy_rmse:.4f} ---")

# 5. Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 6. Evaluate on Validation & Test Data
y_pred_val = lr_model.predict(X_val_scaled)
val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
val_r2 = r2_score(y_val, y_pred_val)

y_pred_test = lr_model.predict(X_test_scaled)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print(f"\n--- Linear Regression Results ---")
print(f"Validation RMSE: {val_rmse:.4f} | R²: {val_r2:.4f}")
print(f"Test RMSE:       {test_rmse:.4f} | R²: {test_r2:.4f}")

# 7. Error Analysis (Residuals)
residuals = y_test - y_pred_test
print(f"\nMean Residual Error (Bias check): {np.mean(residuals):.4f} (should be close to 0)")

Training samples: 14448
Validation samples: 3096
Testing samples: 3096

--- Baseline (Dummy Regressor) RMSE: 1.1499 ---

--- Linear Regression Results ---
Validation RMSE: 0.7354 | R²: 0.5848
Test RMSE:       0.7213 | R²: 0.6066

Mean Residual Error (Bias check): -0.0152 (should be close to 0)



# Experimental Methodology: Linear Regression Pipeline for Housing Price Prediction

## 1. Dataset Selection and Source

* **Dataset Name:** California Housing Dataset.
* **Source:** Acquired programmatically via `scikit-learn` (originally sourced from the 1990 U.S. Census via the StatLib repository).
* **Objective:** Predict the median house value for California districts based on demographic and geographic features (e.g., median income, housing median age, total rooms, population).

## 2. Data Splitting Strategy (Train / Validation / Test)

To ensure rigorous evaluation and prevent overfitting, the dataset was partitioned into three distinct subsets:

* **Training Set (70%):** Used for model parameter estimation and learning coefficients.
* **Validation Set (15%):** Reserved for hyperparameter tuning and intermediate performance evaluation.
* **Testing Set (15%):** Kept completely unseen until final evaluation to assess true out-of-sample generalization.

## 3. Data Preprocessing & Feature Scaling

* **Handling Scale Disparities:** Features in the dataset possessed varying units and numerical ranges (e.g., room counts vs. median income ratios).
* **Standardization (`StandardScaler`):** Features were centered to a mean of $0$ and scaled to a standard deviation of $1$.
* **Prevention of Data Leakage:** To maintain scientific integrity, the scaler was **fitted strictly on the training dataset only** (`fit_transform`), and the parameters derived from training were subsequently applied to transform the validation and test datasets (`transform`).
* 
## 4. Baseline Modeling (Dummy Baseline)

* **Implementation:** A `DummyRegressor` configured with the `"mean"` strategy was established as a baseline.
* **Purpose:** To serve as a benchmark; any effective machine learning model must significantly outperform this naive baseline, which simply predicts the mean target value of the training data.

## 5. Model Training (Supervised Learning)

* **Algorithm:** Ordinary Least Squares Linear Regression.
* **Mechanism:** The algorithm computes a weighted linear combination of input features to minimize the residual sum of squares between the predicted and actual target housing values.

## 6. Evaluation Metrics

The model's predictive performance was quantified using two standard regression metrics:

* **Root Mean Squared Error (RMSE):** Measures the average magnitude of prediction errors in the original units (dollars), penalizing larger errors more heavily.
* **Coefficient of Determination ($R^2$ Score):** Quantifies the proportion of variance in the dependent variable that is predictable from the independent features (where $1.0$ represents a perfect fit).

## 7. Residual Error Analysis

* **Diagnostic Check:** Residuals ($y_{\text{true}} - y_{\text{pred}}$) were analyzed on the test set.
* **Bias Verification:** The mean of the residuals was computed to confirm it approached zero, ensuring the model possesses minimal systematic bias and that errors are symmetrically distributed.# 